# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains code outlining our data collection approach. 
</div>

In [14]:
import requests
import json
import os
import pandas as pd
from pathlib import Path
import warnings
import shelve
import math
from collections import defaultdict
import re

## Data Collection Methodology

1. **Using Google Places API, find nightclubs for a certain area**

2. **Collect up to twenty stations within a radius of 1 kilometre from the nightclub:**
    - allows for up to a twelve minute walk at a typical walking pace of 3mph.

3. **Filter stations according to direction**
    - Preserving the labels from step 2, divide the area surrounding a club into eight 45 degree sections. For each section, take _the two closest stations_ to the nightclub **but always include** train stations (typically larger transport hubs).

4. **Generate Routes:**
    - Use Google Routes API to calculate routes for stations (computeroutematrix allows for batch routing, saving costs).
    Label routes accordingly:
        - 'very short' ( under 5 mins).
        - 'short' (between 5-8 mins).
        - 'long' (between 8-15 mins).

5. **Prepare Routes for Police API:**
    - Reduce the number of route vertices in the google maps polyline;
    - Convert the polyline into a polygon by extending the route polyline on either side, making a 'corridor' to use as a custom area for the police API.

## 1. Gather nightclub data for a popular area in London

This [TimeOut list ](https://www.timeout.com/london/clubs/the-best-clubs-in-london) features 5 clubs in Dalston, the most for any area on the list. This test will use Dalston to generate clubs.

In [22]:
# Constants
PLACES_URL = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')
HEADERS = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': (
        'places.displayName,'
        'places.postalAddress.postalCode,'
        'places.primaryType,'
        'places.location'
    )
}

AREA_NAMES = ['Dalston']
AREA_POSTCODES = {
    'Dalston': ('E8', 'N16')
}


In [25]:
def validate_clubs(area_name, version):
    """
    Queries Google Places API for nightclubs within a given area, 
    validates them by checking that their primaryType is either a bar 
    or a nightclub and that they postcode is correct, caches results, 
    and returns the filtered list.
    """

    # This section tries to call the API, bringing about one of three possible results:
    ## 1. Successful call, cache data is refreshed.
    ## 2. Failed call, previously-cached data is loaded and used safely.
    ## 3. Failed call, no cached data available. Fails loudly

    cache_dir = Path(f'data/raw/{version}')
    cache_path = cache_dir / (f'{area_name.lower()}_nightclubs.json')

    try: 

        query = {
            'textQuery': f'nightclubs in {area_name}, London, England'
        }

        response = requests.post(url=PLACES_URL, headers=HEADERS, data=json.dumps(query))

        response.raise_for_status()
        clubs_data = response.json()
    
    except Exception as e:
        if cache_path.exists():
            warnings.warn(
                f'Places API failed for {area_name}, using cached data instead'
            )
            with open(cache_path) as f:
                return json.load(f)

        else:
            raise RuntimeError(
                f'API failed and no cache available for {area_name}'
            ) from e

    # This section validates clubs returned against their primaryType and postcode

    valid_clubs = []
    postcode_prefixes = AREA_POSTCODES.get(area_name)
    if not postcode_prefixes:
        raise ValueError(f'No postcode prefixes defined for {area_name}')

    for club in clubs_data.get('places', []):
        postcode = (
            club
            .get('postalAddress', {})
            .get('postalCode', '')
        )
        primary_type = club.get("primaryType", "")

        if (
            postcode.startswith(postcode_prefixes)
            and primary_type in {"bar", "nightclub"}
        ):
            valid_clubs.append(club)

    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w') as f:
            json.dump(valid_clubs, f, indent=2)

    return valid_clubs

In [26]:
VERSION = 'v1'
dalston_clubs = validate_clubs('Dalston', VERSION)

## 2. Collect up to twenty stations within a radius of 1 kilometre from the nightclub

In [18]:
# Test constants

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.types,places.formattedAddress,places.location,places.displayName,places.primaryTypeDisplayName',
    }
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [19]:
def slugify(name: str) -> str:
    name = name.lower()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.strip("_")

In [22]:
def get_stations_data(area_name, version):

    clubs_path = Path(f'data/raw/{version}/{area_name.lower()}_nightclubs.json')

    if not clubs_path.exists():
        raise FileNotFoundError(
            f'No nightclub cache found for {area_name} at {clubs_path}'
        )
    
    with open(clubs_path, 'r') as f:
        clubs = json.load(f)

    output_dir = Path(f"data/raw/{version}/{area_name.lower()}")
    output_dir.mkdir(parents=True, exist_ok=True)

    for club in clubs:
        club_name = club['displayName']['text']
        club_slug = slugify(club_name)

        lat = round(club['location']['latitude'], ROUNDING)
        lon = round(club['location']['longitude'], ROUNDING)

        cache_path = output_dir / f'{club_slug}_stations.json'

        if cache_path.exists():
            continue

        data = {
                'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
                'maxResultCount' : 20,
                'rankPreference': 'DISTANCE',
                'locationRestriction' : {
                    'circle': {
                        'center' : {
                            'latitude' : lat,
                            'longitude' : lon
                        },
                        'radius' : 1000
                    }
                }
            }
            

        try: 
            response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data, timeout=10)
            response.raise_for_status()
            stations_data = response.json()

        except Exception as e:
            warnings.warn(
                f'Failed to fetch stations for {club_name}: {e}'
            )
            continue

        output = {
            'area': area_name,
            'club': club_name,
            'club_location': {
                'latitude': lat,
                'longitude': lon,
            },
            'places': stations_data.get('places', []),
        }

        with open(cache_path, 'w') as f:
            json.dump(output, f, indent=2)

In [23]:
VERSION = 'v1'
dalston_stations = get_stations_data('Dalston', VERSION)

## 3. Filter stations according to direction

In [2]:
with open('data/raw/stations_from_dalston_superstore', 'r') as f:
    stations = json.load(f)

In [20]:
stations.keys()

dict_keys(['places'])

In [ ]:
first_club_lat, first_club_long = (
    stations['places'][0]['location']['latitude'],
    stations['places'][0]['location']['longitude']
)

In [23]:
# define a function that returns a bearing fro point 1 to point 2 in degrees (0-360) where 0 = North and 90 = East

def bearing_degrees(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_lon = math.radians(lon2 - lon1)

    x = math.sin(delta_lon) * math.cos(phi2)
    y = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(delta_lon)
    )

    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

In [24]:
# define a function that assigns each station to one of 8 sectors

def sector_index(bearing):
    return int(bearing // 45)

In [4]:
# define haversine to calculate the shortest distance between two points on a sphere

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [66]:
def directional_filtering(stations, club_lat, club_lon, max_per_sector=2):
    sectors = defaultdict(list)
    train_stations = []

    for station in stations['places']:
        lat = station['location']['latitude']
        lon = station['location']['longitude']

        # 1. compute straight-line distance in metres from club to station

        station['distance_m'] = haversine(club_lat, club_lon, lat, lon)

        # 2. assign station to a bearing sector relative to the club

        bearing = bearing_degrees(club_lat, club_lon, lat, lon)
        sector = sector_index(bearing)

        station['bearing'] = bearing
        station['sector'] = sector
        
        # 3. split by station type

        if station['primaryType'] in ('train_station', 'subway_station'):
            train_stations.append(station)
        else:
            sectors[sector].append(station)

    selected = []

    # select closest bus stations per sector

    for sector_stations in sectors.values():
        sector_stations.sort(key=lambda s: s['distance_m'])
        selected.extend(sector_stations[:max_per_sector])

    # always include train/subway stations

    selected_ids = {
        (
            s['location']['latitude'], s['location']['longitude']
            ) 
            for s in selected
        }

    for station in train_stations:
        key = (station['location']['latitude'], station['location']['longitude'])
        if key not in selected_ids:
            selected.append(station)

    return selected

In [ ]:
filtered_dss_stations = directional_filtering(
    stations=stations,
    club_lat=first_club_lat,
    club_lon=first_club_long,
    max_per_sector=2
)

In [ ]:
for station in filtered_dss_stations: 
    print(f'This station is in sector: {station['sector']}')

In [53]:
first_station_lat, first_station_long = (
    filtered_dss_stations[9]['location']['latitude'], 
    filtered_dss_stations[9]['location']['longitude']
)

## 4. Generate Routes for Stations

In [55]:
# Test constants

API_KEY = os.getenv('API_KEY')
ROUTES_URL = 'https://routes.googleapis.com/directions/v2:computeRoutes'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'routes.duration,routes.distanceMeters,routes.polyline',
    }

In [56]:
data = {
    'origin':{
        'location':{
            'latLng':{
                'latitude': first_club_lat,
                'longitude': first_club_long
            }
        }
    },

    'destination':{
        'location':{
            'latLng':{
                'latitude': first_station_lat,
                'longitude': first_station_long
            }
        }
    },

    'travelMode': 'WALK',
    'polylineEncoding': 'GEO_JSON_LINESTRING',
    'polylineQuality': 'OVERVIEW'
}

response = requests.post(url=ROUTES_URL, headers=HEADERS, json=data)

In [59]:
with open('data/raw/test_routes_data.json', 'w') as f:
    json.dump(response.json(), f, indent=2)

## 5. Prepare Routes for Police API

In [5]:
with open('data/raw/test_routes_data.json', 'r') as f:
    routes = json.load(f)

In [8]:
def reduce_polyline(route, min_spacing_m):
# This function slims down the polyline provided by google maps api to reduce number of route vertices for Police API
    polyline = route['routes'][0]['polyline']['geoJsonLinestring']['coordinates']
    sampled = [polyline[0]]
    acc_dist = 0.0
    prev_lon, prev_lat = polyline[0]

    for curr_lon, curr_lat in polyline[1:]:
        step_distance = haversine(prev_lat, prev_lon, curr_lat, curr_lon)
        acc_dist += step_distance

        if acc_dist >= min_spacing_m:
            sampled.append([curr_lon, curr_lat])
            acc_dist = 0.0

        prev_lon, prev_lat = curr_lon, curr_lat

    if sampled[-1] != polyline[-1]:
        sampled.append(polyline[-1])

    return sampled

In [9]:
test_reduce = reduce_polyline(routes, 30)
print(len(test_reduce))
print(len(routes['routes'][0]['polyline']['geoJsonLinestring']['coordinates']))

13
21


In [ ]:
def create_corridor(polyline, dist_ext):
    left_side = []
    right_side = []

    for i in range(len(polyline) - 1):
        lon1, lat1 = polyline[i]
        lon2, lat2 = polyline [i+1]

        metres_per_deg_lat = 111_320
        metres_per_deg_lon = 111_320 * math.cos(math.radians(lat1))

        dx = lon2-lon1
        dy= lat2-lat1

        length = math.hypot(dx * metres_per_deg_lon, dy * metres_per_deg_lat)
        if length == 0:
            continue

        # Unit perpendicular vector
        ux = -dy * metres_per_deg_lat / length
        uy =  dx * metres_per_deg_lon / length

        # Offset in degrees
        offset_lon = (ux * dist_ext) / metres_per_deg_lon
        offset_lat = (uy * dist_ext) / metres_per_deg_lat

        # Left and right offsets
        left_side.append([lat1 + offset_lat, lon1 + offset_lon])
        right_side.append([lat1 - offset_lat, lon1 - offset_lon])

    # Handle final point
    lon_end, lat_end = polyline[-1]
    left_side.append([lat_end + offset_lat, lon_end + offset_lon])
    right_side.append([lat_end - offset_lat, lon_end - offset_lon])

    # Combine into closed polygon
    polygon = left_side + right_side[::-1]
    polygon.append(polygon[0])

    return polygon

In [11]:
test_corridor = create_corridoor(test_reduce, dist_ext=25)

In [12]:
def polygon_to_poly_param(polygon):
    return ":".join(f"{lat},{lon}" for lat, lon in polygon)

In [13]:
poly_param = polygon_to_poly_param(test_corridor)

## 6. Request Police API

In [20]:
POLICE_URL = "https://data.police.uk/api/crimes-street/all-crime"

params = {
    'date': '2024-05',
    'poly': poly_param,
}

response = requests.get(POLICE_URL, params=params)

In [21]:
crimes = response.json()

allowed_categories = {
    "anti-social-behaviour",
    "violent-crime",
    "public-order"
}

filtered_crimes = [
    c for c in crimes
    if c["category"] in allowed_categories
]


In [24]:
with open('data/raw/police_api_test.json', 'w') as f:
    json.dump(filtered_crimes, f, indent=2)

## Function Design

### Limitations and Reflections before NB02

_Data Quality of Police API_

The [Police API](https://data.police.uk/about/) lists some known issues regarding data quality. A particularly important issue for our purposes is that there are inconsistent geocoding policies in police forces. Estimates of geocoding accuracy in different forces range from 60% to 97%. We cannot filter for confident location data so we will acknowledge this limitation in our report.

_Police API Limitations_

The Police API does not allow for requests at a specific time; it only gives the month that the crime was commited. This will be acknowledged in our report.

_Google Route Matrix API Version_

The 'Walk' option of the [RouteTravelMode](https://developers.google.com/maps/documentation/routes/reference/rest/v2/RouteTravelMode?_gl=1*1v9v6vk*_up*MQ..*_ga*MTQ3MDM2MzQyLjE3NjcyMzk3MzQ.*_ga_NRWSTWS78N*czE3NjcyMzk3MzMkbzEkZzEkdDE3NjcyNDAyMTkkajYwJGwwJGgw*_ga_SM8HXJ53K2*czE3NjcyMzk3MzMkbzEkZzAkdDE3NjcyNDAyMTckajYwJGwwJGgw) is in beta. Google Maps state that it might sometimes be missing clear sidewalk and pedestrian paths. 

_Justifications_

The **downsampling** of the google routes polyline was necessary because the corridor polygon that we later compute creates more vertices (two opposite coordinates perpendicular to the polyline coordinates). Limiting the number of vertices is necessary for good API citizenship when requesting the Police API. 

The 'buffer polygon' created in part 4 maps the _actual route taken_: it is not simply an arbitrary box containing coordinates along the route.